In [ ]:
#!/usr/bin/env python
import numpy as np
import pandas as pd
import torch
from torch import nn
import math
from einops import rearrange
import numpy as np
import os
import sys
import json
import time
import matplotlib.pyplot as plt
import configparser

# %%
class ModelParams:
    
    @property
    def d_model(self):
        return self._d_model
    
    @property
    def s_conv_kernel(self):
        return self._s_conv_kernel
    
    @property
    def l_conv_kernel(self):
        return self._l_conv_kernel
    
    @property
    def max_seq_length(self):
        return self._max_seq_length
    
    @property
    def emb_base(self):
        return self._emb_base
    
    @property
    def dtype(self):
        return self._dtype
    
    @property
    def d_ff(self):
        return self._d_ff
    
    @property
    def src_vocb_size(self):
        return self._src_vocb_size
    
    @property
    def tgt_vocb_size(self):
        return self._tgt_vocb_size
    
    @property
    def num_heads(self):
        return self._num_heads
    
    @property
    def order(self):
        return self._order
    
    @property
    def emb_dim(self):
        return self._emb_dim
    
    # new properties for original HyenaEncoder
    
    
    def __init__(self, 
                 src_vocb_size: int,
                 tgt_vocb_size: int,
                 d_model: int, 
                 num_heads: int,
                 max_seq_length: int,
                 s_conv_kernel: int = 3, 
                 l_conv_kernel: int = 21, 
                 emb_base: int = 10000, 
                 d_ff: int = 512,
                 dtype = torch.float,
                 order = int,
                 emb_dim = int):
        assert d_model % num_heads == 0 and d_model % 2 == 0 and num_heads % 2 == 0, 'd_model must be divisible by num_heads and both d_model and num_heads must be even.\n'
        self._src_vocb_size = src_vocb_size
        self._tgt_vocb_size = tgt_vocb_size
        self._d_model = d_model
        self._num_heads = num_heads
        self._s_conv_kernel = s_conv_kernel
        self._l_conv_kernel = l_conv_kernel
        self._max_seq_length = max_seq_length
        self._emb_base = emb_base
        self._dtype = dtype
        self._d_ff = d_ff
        self._order = order
        self._emb_dim = emb_dim
        
        # Mutable variables
        self.conv_kernel = s_conv_kernel

def get_params(config_path):
    config = configparser.ConfigParser()
    config.read(config_path)
    params = ModelParams(
        src_vocb_size = config.getint('model_architecture', 'src_vocb_size'),
        tgt_vocb_size = config.getint('model_architecture', 'tgt_vocb_size'),
        d_model = config.getint('model_architecture', 'd_model'), 
        num_heads = config.getint('model_architecture', 'num_heads'),
        max_seq_length = config.getint('model_architecture', 'max_seq_length'),
        s_conv_kernel = config.getint('model_architecture', 's_conv_kernel'), 
        l_conv_kernel = config.getint('model_architecture', 'l_conv_kernel'), 
        emb_base = config.getint('model_architecture', 'emb_base'), 
        d_ff = config.getint('model_architecture', 'd_ff'),
        dtype = eval(config.get('model_architecture', 'dtype')),
        order = config.getint('model_architecture', 'order'),
        emb_dim = config.getint('model_architecture', 'emb_dim')
    )
    return params
        
# class and method definition
class Hyena(nn.Module):
    # Modified from UniversalTools to fix inaccuracies of orig script
    def __init__(self, params: ModelParams):
        super(Hyena, self).__init__()
        # channel mix layers, dense layer for each column
        self.W_q = nn.Linear(params.d_model, params.d_model) # Query transformation
        self.W_k = nn.Linear(params.d_model, params.d_model) # Key transformation
        self.W_v = nn.Linear(params.d_model, params.d_model) # Value transformation
    
        # short convolution
        self.conv_q = nn.Conv1d(params.d_model, params.d_model, params.s_conv_kernel, stride=1, padding=params.s_conv_kernel//2)
        self.conv_k = nn.Conv1d(params.d_model, params.d_model, params.s_conv_kernel, stride=1, padding=params.s_conv_kernel//2)
        self.conv_v = nn.Conv1d(params.d_model, params.d_model, params.s_conv_kernel, stride=1, padding=params.s_conv_kernel//2)
    
        # gate function
        self.glu = nn.GLU(-1)
        
        # long convolution
        self.conv_l = nn.Conv1d(params.d_model, params.d_model, params.l_conv_kernel, stride=1, padding=params.l_conv_kernel//2)
    
    def forward(self, x):
        """
        X should be sized as (B, L, d) or (L, d)
        
        x => Q, K, V
        result: V x \sigma(Q x \sigma(K))
        """
        # channel mix: (B, L, d) => (B, L, d)
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
    
        # sequnce mix: (B, L, d) => (B, d, L) => (B, d, L) => (B, L, d)
        cq = self.conv_q(q.transpose(-2,-1)).transpose(-2,-1)
        ck = self.conv_k(k.transpose(-2,-1)).transpose(-2,-1)
        cv = self.conv_v(v.transpose(-2,-1)).transpose(-2,-1)
    
        # stack gq and gk: (B, L, d) + (B, L, d) => (B, L, 2d)
        mix_qk = torch.cat([cq, ck], dim=-1)
        
        # gate Q, K: (B, L, 2d) => (B, L, d)
        g_qk = self.glu(mix_qk)
    
        # long distance convolution: (B, L, d) => (B, d, L) => (B, d, L) => (B, L, d)
        l_qk = self.conv_l(g_qk.transpose(-2,-1)).transpose(-2,-1)
    
        # stack gq and gk: (B, L, d) + (B, L, d) => (B, L, 2d)
        mix_qkv = torch.cat([cv,l_qk], dim=-1)
        
        # gate on mix_qkv: (B, L, 2d) => (B, L, d)
        
        # print("Hyena operation completed")
        
        return self.glu(mix_qkv)

# fixed with proper weight initialization
class SwiGLU(nn.Module):
    def __init__(self, input_dim, output_dim=None):
        super().__init__()
        d = input_dim // 2
        self.linear_v = nn.Linear(d, d)
        self.output_proj = nn.Linear(d, d if output_dim is None else output_dim)
        # Proper initialization
        nn.init.xavier_uniform_(self.linear_v.weight)
        nn.init.zeros_(self.linear_v.bias)
        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, x):
        x1, x2 = torch.chunk(x, 2, dim=-1)
        swish = x1 * torch.sigmoid(x1)
        out = swish * self.linear_v(x2)
        return self.output_proj(out)

class SwiGLUHyena(nn.Module):
    # Modified from UniversalTools to fix inaccuracies of orig script
    def __init__(self, params: ModelParams):
        super(SwiGLUHyena, self).__init__()
    
        # input x should be sized as B x L x d
        # where B is Batch size
        # L is length of the input sequence
        # d is dimension of a token
        
        # channel mix layers, dense layer for each column
        self.W_q = nn.Linear(params.d_model, params.d_model) # Query transformation
        self.W_k = nn.Linear(params.d_model, params.d_model) # Key transformation
        self.W_v = nn.Linear(params.d_model, params.d_model) # Value transformation
    
        # short convolution
        self.conv_q = nn.Conv1d(params.d_model, params.d_model, params.s_conv_kernel, stride=1, padding=params.s_conv_kernel//2)
        self.conv_k = nn.Conv1d(params.d_model, params.d_model, params.s_conv_kernel, stride=1, padding=params.s_conv_kernel//2)
        self.conv_v = nn.Conv1d(params.d_model, params.d_model, params.s_conv_kernel, stride=1, padding=params.s_conv_kernel//2)
    
        # gate function
        self.swiglu = SwiGLU(input_dim=2*params.d_model, output_dim=params.d_model)
        # self.glu = nn.GLU(-1)
        
        # long convolution
        self.conv_l = nn.Conv1d(params.d_model, params.d_model, params.l_conv_kernel, stride=1, padding=params.l_conv_kernel//2)
        # layernorm
        self.norm_qk = nn.LayerNorm(2 * params.d_model)
        self.norm_qkv = nn.LayerNorm(2 * params.d_model)
    
    def forward(self, x):
        """
        X should be sized as (B, L, d) or (L, d)
        
        x => Q, K, V
        result: V x \sigma(Q x \sigma(K))
        """
        # channel mix: (B, L, d) => (B, L, d)
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
    
        # sequnce mix: (B, L, d) => (B, d, L) => (B, d, L) => (B, L, d)
        cq = self.conv_q(q.transpose(-2,-1)).transpose(-2,-1)
        ck = self.conv_k(k.transpose(-2,-1)).transpose(-2,-1)
        cv = self.conv_v(v.transpose(-2,-1)).transpose(-2,-1)
    
        # stack gq and gk: (B, L, d) + (B, L, d) => (B, L, 2d)
        mix_qk = torch.cat([cq, ck], dim=-1)
        
        # gate Q, K: (B, L, 2d) => (B, L, d)
        mix_qk = self.norm_qk(mix_qk) # layernorm after
        g_qk = self.swiglu(mix_qk)
        # g_qk = self.glu(mix_qk)
    
        # long distance convolution: (B, L, d) => (B, d, L) => (B, d, L) => (B, L, d)
        l_qk = self.conv_l(g_qk.transpose(-2,-1)).transpose(-2,-1)
    
        # stack gq and gk: (B, L, d) + (B, L, d) => (B, L, 2d)
        mix_qkv = torch.cat([cv,l_qk], dim=-1)
        
        # gate on mix_qkv: (B, L, 2d) => (B, L, d)
        
        # print("Hyena operation completed")
        mix_qkv = self.norm_qkv(mix_qkv) # layernorm after (added)
        return self.swiglu(mix_qkv)
######################################################################################
# Hyena model from original publication
def fftconv(u, k, D):
    seqlen = u.shape[-1]
    fft_size = 2 * seqlen
    
    k_f = torch.fft.rfft(k, n=fft_size) / fft_size
    u_f = torch.fft.rfft(u.to(dtype=k.dtype), n=fft_size)
    
    if len(u.shape) > 3: k_f = k_f.unsqueeze(1)
    y = torch.fft.irfft(u_f * k_f, n=fft_size, norm='forward')[..., :seqlen]

    out = y + u * D.unsqueeze(-1)
    return out.to(dtype=u.dtype)

# =======start of hyena operator=======
@torch.jit.script 
def mul_sum(q, y):
    return (q * y).sum(dim=1)

class OptimModule(nn.Module):
    """ Interface for Module that allows registering buffers/parameters with configurable optimizer hyperparameters """

    def register(self, name, tensor, lr=None, wd=0.0):
        """Register a tensor with a configurable learning rate and 0 weight decay"""

        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))

            optim = {}
            if lr is not None: optim["lr"] = lr
            if wd is not None: optim["weight_decay"] = wd
            setattr(getattr(self, name), "_optim", optim)
            

class Sin(nn.Module):
    def __init__(self, dim, w=10, train_freq=True):
        super().__init__()
        self.freq = nn.Parameter(w * torch.ones(1, dim)) if train_freq else w * torch.ones(1, dim)

    def forward(self, x):
        return torch.sin(self.freq * x)
    
    
class PositionalEmbedding(OptimModule):
    def __init__(self, emb_dim: int, seq_len: int, lr_pos_emb: float=1e-5, **kwargs): 
        """Complex exponential positional embeddings for Hyena filters."""  
        super().__init__()
        
        self.seq_len = seq_len
        # The time embedding fed to the filteres is normalized so that t_f = 1
        t = torch.linspace(0, 1, self.seq_len)[None, :, None] # 1, L, 1
        
        if emb_dim > 1:
            bands = (emb_dim - 1) // 2            
        # To compute the right embeddings we use the "proper" linspace 
        t_rescaled = torch.linspace(0, seq_len - 1, seq_len)[None, :, None]
        w = 2 * math.pi * t_rescaled / seq_len # 1, L, 1 
        
        f = torch.linspace(1e-4, bands - 1, bands)[None, None] 
        z = torch.exp(-1j * f * w)
        z = torch.cat([t, z.real, z.imag], dim=-1)
        self.register("z", z, lr=lr_pos_emb) 
        self.register("t", t, lr=0.0)
        
    def forward(self, L):
        return self.z[:, :L], self.t[:, :L]
    

class ExponentialModulation(OptimModule):
    def __init__(
        self,
        d_model,
        fast_decay_pct=0.3,
        slow_decay_pct=1.5,
        target=1e-2,
        modulation_lr=0.0,
        modulate: bool=True,
        shift: float = 0.0,
        **kwargs
    ):
        super().__init__()
        self.modulate = modulate
        self.shift = shift
        max_decay = math.log(target) / fast_decay_pct
        min_decay = math.log(target) / slow_decay_pct
        deltas = torch.linspace(min_decay, max_decay, d_model)[None, None]
        self.register("deltas", deltas, lr=modulation_lr)
        
    def forward(self, t, x):
        if self.modulate:
            decay = torch.exp(-t * self.deltas.abs()) 
            x = x * (decay + self.shift)
        return x                  

class HyenaFilter(nn.Module):
    def __init__(
        self, 
        d_model: int,            # Model dimension
        order: int = 16,         # Order of the filter
        emb_dim: int = 3,        # Embedding dimension
        seq_len: int = 285,      # Maximum sequence length
        lr_pos_emb: float = 1e-5, # Learning rate for positional embeddings
        modulation_lr: float = 0.0, # Learning rate for modulation
        modulate: bool = True,    # Enable/disable modulation
        shift: float = 0.0,       # Shift value for modulation
        num_inner_mlps: int = 2,  # Number of inner MLPs
        use_bias: bool = True,     # Use bias
        dropout: float = 0.0       # Dropout rate
    ):
        """
        Implicit long filter with modulation.

        Args:
            params (ModelParams): Configuration parameters containing various hyperparameters
            lr: Learning rate for the positional embeddings
            modulation_lr: Learning rate for the modulation
            modulate: Boolean flag to enable/disable modulation
            shift: Shift value for modulation
        """
        super().__init__()
        self.d_model = d_model
        self.use_bias = use_bias
        self.dropout = nn.Dropout(dropout)
        self.order = order
        self.emb_dim = emb_dim
        self.seq_len = seq_len
        
        self.bias = nn.Parameter(torch.randn(self.d_model)) if self.use_bias else None
        
        act = Sin(dim=self.order, w=1)  # Assume Sin is defined elsewhere
        self.pos_emb = PositionalEmbedding(self.emb_dim, self.seq_len, lr_pos_emb=lr_pos_emb)  # Assume PositionalEmbedding is defined
        
        # Construct the implicit filter architecture
        self.implicit_filter = nn.Sequential(
            nn.Linear(self.emb_dim, self.order),
            act,
        )
        for _ in range(num_inner_mlps):
            self.implicit_filter.append(nn.Linear(self.order, self.order))
            self.implicit_filter.append(act)

        self.implicit_filter.append(nn.Linear(self.order, self.d_model, bias=False))
        
        self.modulation = ExponentialModulation(self.d_model)  # Assume ExponentialModulation is defined

    def filter(self, L):
        z, t = self.pos_emb(L)
        h = self.implicit_filter(z)
        h = self.modulation(t, h)
        return h

    def forward(self, x, L, k=None, bias=None):
        if k is None: 
            k = self.filter(L)

        # Ensure compatibility with filters that return a tuple 
        k = k[0] if isinstance(k, tuple) else k 

        y = fftconv(x, k, bias)  # Assume fftconv is defined
        return y

        
class HyenaOperator(nn.Module):
    def __init__(self, params: ModelParams, **filter_args):
        r"""
        Hyena operator described in the paper https://arxiv.org/pdf/2302.10866.pdf
        
        Args:
            params (ModelParams): Configuration parameters
            filter_args: Additional arguments for the HyenaFilter
        """
        super().__init__()
        self.d_model = params.d_model
        self.l_max = params.max_seq_length
        self.order = 2  # This can also be configurable
        inner_width = self.d_model * (self.order + 1)
        self.dropout = nn.Dropout(0.0)  # Set default dropout; make configurable if needed
        self.in_proj = nn.Linear(self.d_model, inner_width)
        self.out_proj = nn.Linear(self.d_model, self.d_model)
        
        self.short_filter = nn.Conv1d(
            inner_width, 
            inner_width, 
            params.s_conv_kernel,
            padding=params.s_conv_kernel // 2,
            groups=inner_width
        )

        self.filter_fn = HyenaFilter(
            d_model=self.d_model * (self.order - 1),
            order=filter_args.get('filter_order', 64),  # Use default if not provided
            seq_len=self.l_max,
            dropout=filter_args.get('filter_dropout', 0.0),  # Use default if not provided
            **filter_args
        )

    def forward(self, u, *args, **kwargs):
        l = u.size(-2)
        l_filter = min(l, self.l_max)
        u = self.in_proj(u)
        u = rearrange(u, 'b l d -> b d l')
        
        uc = self.short_filter(u)[...,:l_filter] 
        *x, v = uc.split(self.d_model, dim=1)
        
        k = self.filter_fn.filter(l_filter)[0]
        k = rearrange(k, 'l (o d) -> o d l', o=self.order - 1)
        bias = rearrange(self.filter_fn.bias, '(o d) -> o d', o=self.order - 1)
        
        for o, x_i in enumerate(reversed(x[1:])):
            v = self.dropout(v * x_i)
            v = self.filter_fn(v, l_filter, k=k[o], bias=bias[o])

        y = rearrange(v * x[0], 'b d l -> b l d')

        y = self.out_proj(y)
        
        return y
# =======End of hyena operator=======

######################################################################################

class SmoothConv(nn.Module):
    def __init__(self, params: ModelParams):
        super(SmoothConv, self).__init__()
        
        d_model = params.d_model
        conv_kernel = params.s_conv_kernel
        d_ff = params.d_ff
        
        self.conv = nn.Conv1d(d_model, d_model, conv_kernel, stride=1, padding=conv_kernel//2)
        self.ff = PositionWiseFeedForward(params)    
        
    def forward(self, x):   
        x = x + self.conv(x.transpose(-2,-1)).transpose(-2,-1)
        return x + self.ff(x)

    
class LSmoothConv(nn.Module):
    def __init__(self, params: ModelParams):
        super(LSmoothConv, self).__init__()
        
        d_model = params.d_model
        conv_kernel = params.l_conv_kernel
        d_ff = params.d_ff

        self.conv = nn.Conv1d(d_model, d_model, conv_kernel, stride=1, padding=conv_kernel//2)
        self.ff = PositionWiseFeedForward(params)                 
        
    def forward(self, x):   
        x = x + self.conv(x.transpose(-2,-1)).transpose(-2,-1)
        return x + self.ff(x)
    
    
class AsynAbsolutePositionalEncoding(nn.Module):
    def __init__(self, params: ModelParams):
        super(AsynAbsolutePositionalEncoding, self).__init__()
        
        pe = torch.zeros(params.max_seq_length, params.d_model)
        
        position = torch.arange(0, params.max_seq_length, dtype=params.dtype).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, params.d_model, 2).float() * -(np.log(params.emb_base) / params.d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x: torch.tensor):
        return x + self.pe[:, :x.size(1)]

class SemiSynAbsolutePositionalEncoding(nn.Module):
    def __init__(self, params: ModelParams):
        super(SemiSynAbsolutePositionalEncoding, self).__init__()
        pe = torch.zeros(params.max_seq_length, params.d_model)
        theta = torch.pi / params.max_seq_length
        head_width = params.d_model // params.num_heads
        thetas = torch.tensor([theta / 2**(i//head_width) for i in range(0, params.d_model, 2)], dtype=params.dtype)
        position = torch.arange(0, params.max_seq_length, dtype=params.dtype).unsqueeze(1)
        
        pe[:, 0::2] = torch.sin(position * thetas)
        pe[:, 1::2] = torch.cos(position * thetas)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x: torch.tensor):
        return x + self.pe[:, :x.size(1)]
    
class SynAbsolutePositionalEncoding(nn.Module):
    def __init__(self, params: ModelParams):
        super(SynAbsolutePositionalEncoding, self).__init__()
        
        pe = torch.zeros(params.max_seq_length, params.d_model)
        theta = torch.pi / params.max_seq_length
        thetas = torch.ensor([theta for _ in range(0, params.d_model, 2)], dtype=params.dtype)
        position = torch.arange(0, params.max_seq_length, dtype=params.dtype).unsqueeze(1)
        
        pe[:, 0::2] = torch.sin(position * thetas)
        pe[:, 1::2] = torch.cos(position * thetas)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x: torch.tensor):
        return x + self.pe[:, :x.size(1)]

class SynRotationalPositionalEncoding(nn.Module):
    """
    Synchronous Rotational Position Encoding
    
    a high dimension tensor can split into a set of 2d vectors, noted as X_i where i in [0, d//2], 
    and this class will apply same rotation effects (angle theta) to all X_i
    
    In other word, the rotation angle theta only chage by the position.
    a high dimension tensor at position p will be modified by the theta_p
    
    theta_p = p * theta

    where theta = pi / max_seq_length
    """
    def __init__(self, params: ModelParams):
        super(SynRotationalPositionalEncoding, self).__init__()
        theta = torch.pi / params.max_seq_length
        pe = torch.tensor([[[np.cos(idx*theta), -np.sin(idx*theta)],[np.sin(idx*theta), np.cos(idx*theta)]] for idx in range(params.max_seq_length)], dtype=params.dtype)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.tensor):
        shapes = x.size()
        if len(shapes) == 3:
            b, l, d = x.size()
            trans_rz = x.view(b, l, int(d/2), 2) @ self.pe[0:l,:,:].transpose(-2,-1)
            return trans_rz.view(b, l, d)
        elif len(shapes) == 2:
            l, d = x.size()
            trans_rz = x.view(l, int(d/2), 2) @ self.pe[0:l,:,:].transpose(-2,-1)
            return trans_rz.view(l, d)
        else:
            raise Exception('The input with wrong shape\n')

class AsynRotationalPositionalEncoding(nn.Module):
    """
    Asynchronous Rotational Position Encoding

    a high dimension tensor can split into a set of 2d vectors, noted as X_i where i in [0, d//2], 
    and this class will apply different rotation effects (angle theta_i) to X_i
    
    In other word, the rotation angle theta chage by both the position and dimension index.
    a high dimension tensor at position p will be modified by the theta_p
    
    theta_p = p * theta_i

    where theta_i = 1/emb_base^(i/d), i in [0, d//2]
    """
    def __init__(self, params: ModelParams):
        super(AsynRotationalPositionalEncoding, self).__init__()
        
        pe = np.zeros((params.max_seq_length, params.d_model, params.d_model))
        for i in range(params.max_seq_length):
            for d in range(params.d_model//2):
                theta = i * np.exp(2*d * -(np.log(params.emb_base) / params.d_model))
                d = d * 2
                pe[i,d,d] = np.cos(theta)
                pe[i,d+1,d] = -np.sin(theta)
                pe[i,d,d+1] = np.sin(theta)
                pe[i,d+1,d+1] = np.cos(theta)
        pe = torch.tensor(pe, dtype=params.dtype)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.tensor):
        shapes = x.size()
        if len(shapes) == 3:
            b, l, d = x.size()
            rz = x.view(b,l,1,d) @ self.pe[:l,:,:]
            return rz.view(b, l, d)
        elif len(shapes) == 2:
            l, d = x.size()
            rz = x.view(l,1,d) @ self.pe[:l,:,:]
            return rz.view(l,d)
        else:
            raise Exception('The input with wrong shape\n')

class SemiSynRotationalPositionalEncoding(nn.Module):
    """
    Asynchronous Rotational Position Encoding

    a high dimension tensor can split into a set of 2d vectors, noted as X_i where i in [0, d//2], 
    and this class will apply different rotation effects (angle theta_i) to X_i
    
    In other word, the rotation angle theta chage by both the position and dimension index.
    a high dimension tensor at position p will be modified by the theta_p
    
    theta_p = p * theta_i

    where theta_i = 1/emb_base^(i/d), i in [0, d//2]
    """
    def __init__(self, params: ModelParams):
        super(SemiSynRotationalPositionalEncoding, self).__init__()        
        pe = np.zeros((params.max_seq_length, params.d_model, params.d_model))
        theta_base = torch.pi / params.max_seq_length
        head_width = params.d_model / params.num_heads
        
        for i in range(params.max_seq_length):
            for d in range(0, params.d_model, 2):
                theta = i * theta_base / 2**(d//head_width)
                pe[i,d,d] = np.cos(theta)
                pe[i,d+1,d] = -np.sin(theta)
                pe[i,d,d+1] = np.sin(theta)
                pe[i,d+1,d+1] = np.cos(theta)
        pe = torch.tensor(pe, dtype=params.dtype)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.tensor):
        shapes = x.size()
        if len(shapes) == 3:
            b, l, d = x.size()
            rz = x.view(b,l,1,d) @ self.pe[:l,:,:]
            return rz.view(b, l, d)
        elif len(shapes) == 2:
            l, d = x.size()
            rz = x.view(l,1,d) @ self.pe[:l,:,:]
            return rz.view(l,d)
        else:
            raise Exception('The input with wrong shape\n')

class PositionWiseFeedForward(nn.Module):
    def __init__(self, params: ModelParams):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(params.d_model, params.d_ff)
        self.fc2 = nn.Linear(params.d_ff, params.d_model)
        self.relu = nn.ReLU()

    def forward(self, x: torch.tensor):
        return self.fc2(self.relu(self.fc1(x)))

class SwiGLU_ff(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        # input_dim should be 2*d_ff
        self.linear_v = nn.Linear(input_dim // 2, input_dim // 2)
        self.output_proj = nn.Linear(input_dim // 2, output_dim)

        # Proper initialization (Xavier for linear layers)
        nn.init.xavier_uniform_(self.linear_v.weight)
        nn.init.zeros_(self.linear_v.bias)
        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, x):
        # x shape: (B, L, 2*d_ff)
        x1, x2 = torch.chunk(x, 2, dim=-1)
        swish = x1 * torch.sigmoid(x1)
        out = swish * self.linear_v(x2)
        return self.output_proj(out)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, params: ModelParams):
        super(SwiGLUFeedForward, self).__init__()
        # Project up to 2*d_ff for SwiGLU
        self.fc1 = nn.Linear(params.d_model, 2 * params.d_ff)
        self.swiglu = SwiGLU_ff(2 * params.d_ff, params.d_model)
        self.norm = nn.LayerNorm(params.d_model)

        # Proper initialization
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)

    def forward(self, x: torch.tensor):
        x = self.norm(x)
        return self.swiglu(self.fc1(x))
    
def place_holder(x):
    return x

class PositionalAttention(nn.Module):
    def __init__(self, params: ModelParams):
        super(PositionalAttention, self).__init__()
        # Initialize dimensions
        self.d_model = params.d_model # Model's dimension
        self.num_heads = params.num_heads # Number of attention heads
        self.d_k = params.d_model // params.num_heads # Dimension of each head's key, query, and value
        
        self.posE = place_holder
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(params.d_model, params.d_model) # Query transformation
        self.W_k = nn.Linear(params.d_model, params.d_model) # Key transformation
        self.W_v = nn.Linear(params.d_model, params.d_model) # Value transformation
        self.W_o = nn.Linear(params.d_model, params.d_model) # Output transformation

    def scaled_dot_product_attention(self, Q: torch.tensor, K: torch.tensor, V: torch.tensor, mask: torch.tensor=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x: torch.tensor):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x: torch.tensor):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, x: torch.tensor):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.posE(self.W_q(x)))
        K = self.split_heads(self.posE(self.W_k(x)))
        V = self.split_heads(self.W_v(x))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, None)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

class SynRotaryAttention(PositionalAttention):

    def __init__(self, params: ModelParams):
        super(SynRotaryAttention, self).__init__(params)
        self.posE = SynRotationalPositionalEncoding(params)

class SemiSynRotaryAttention(PositionalAttention):

    def __init__(self, params: ModelParams):
        super(SemiSynRotaryAttention, self).__init__(params)
        self.posE = SemiSynRotationalPositionalEncoding(params)
        
        
class AsynRotaryAttention(PositionalAttention):

    def __init__(self, params: ModelParams):
        super(AsynRotaryAttention, self).__init__(params)
        self.posE = AsynRotationalPositionalEncoding(params)

class SynAbsoluteAttention(PositionalAttention):

    def __init__(self, params: ModelParams):
        super(SynAbsoluteAttention, self).__init__(params)
        self.posE = SynAbsolutePositionalEncoding(params)

class SemiSynAbsoluteAttention(PositionalAttention):

    def __init__(self, params: ModelParams):
        super(SemiSynAbsoluteAttention, self).__init__(params)
        self.posE = SemiSynAbsolutePositionalEncoding(params)
        
class AsynAbsoluteAttention(PositionalAttention):

    def __init__(self, params: ModelParams):
        super(AsynAbsoluteAttention, self).__init__(params)
        self.posE = AsynAbsolutePositionalEncoding(params)

class AttnEncoder(nn.Module):
    def __init__(self, params: ModelParams):
        super(AttnEncoder, self).__init__()
        self.attn = AsynAbsoluteAttention(params)
        self.ff = PositionWiseFeedForward(params)
    
    def forward(self, x: torch.tensor):
        x = x + self.attn(x)
        return x + self.ff(x)
    
class RAttnEncoder(nn.Module):
    def __init__(self, params: ModelParams):
        super(RAttnEncoder, self).__init__()
        self.attn = AsynRotaryAttention(params)
        self.ff = PositionWiseFeedForward(params)
    
    def forward(self, x: torch.tensor):
        x = x + self.attn(x)
        return x + self.ff(x)
class SwiGLUAttnEncoder(nn.Module):
    def __init__(self, params: ModelParams):
        super(SwiGLUAttnEncoder, self).__init__()
        self.attn = AsynAbsoluteAttention(params)
        # self.ff = PositionWiseFeedForward(params)
        self.norm1 = nn.LayerNorm(params.d_model)
        self.ff = SwiGLUFeedForward(params)
        self.norm2 = nn.LayerNorm(params.d_model)
    
    def forward(self, x: torch.tensor):
        # LayerNorm before attention
        x = x + self.attn(self.norm1(x))
        # LayerNorm before feedforward
        x = x + self.ff(self.norm2(x))
        return x

class SwiGLURAttnEncoder(nn.Module):
    """
    Main attention encoder block with SwiGLU feedforward and LayerNorm.
    """
    def __init__(self, params: ModelParams):
        super(SwiGLURAttnEncoder, self).__init__()
        self.attn = AsynRotaryAttention(params)
        self.norm1 = nn.LayerNorm(params.d_model)
        self.ff = SwiGLUFeedForward(params)
        self.norm2 = nn.LayerNorm(params.d_model)

    def forward(self, x: torch.tensor):
        # LayerNorm before attention
        x = x + self.attn(self.norm1(x))
        # LayerNorm before feedforward
        x = x + self.ff(self.norm2(x))
        return x
        
lidx_layers = {
    1: Hyena,
    2: SmoothConv,
    3: LSmoothConv,
    4: AttnEncoder,
    5: RAttnEncoder,
    6: HyenaOperator,
    7: SwiGLUHyena,
    8: SwiGLUAttnEncoder,
    9: SwiGLURAttnEncoder
}

class TrainModel:
    
    def __init__(self, 
                 model, 
                 loss_func, 
                 optimizer, 
                 epoch_num, 
                 batch_size, 
                 params_path, 
                 save_span, 
                 data_train, 
                 data_test, 
                 show_progress):
        self._model = model
        self._loss_func = loss_func
        self._optimizer = optimizer
        self._epoch_num = epoch_num
        self._batch_size = batch_size
        self._output_path = params_path
        self._save_span = save_span
        self._data_train = data_train
        self._data_test = data_test
        self._show_progress = show_progress
        self._trained = False
        
        self.log = {'epoch': [],
                    'train': [],
                    'test': [],
                    # 'lr': []
                   }
    
    def train(self):
        
        record_num_train = len(self._data_train['x'])
        record_num_test = len(self._data_test['x'])
        
        # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self._optimizer, mode='min', factor=0.66, patience=5)
        for epoch in range(self._epoch_num):
            log_loss = {'train': [], 'test': []}
            self._model.train()
            # torch.autograd.set_detect_anomaly(True)
            for i in range(0, record_num_train, self._batch_size):
                # sys.stderr.write(f'\r{time.asctime()} Train Progress: {i} - {len(Xt)}')
                self._optimizer.zero_grad()
                loss = None
                e_size = 0
                for idx in range(self._batch_size):
                    if idx + i >= record_num_train:
                        break
                    e_size += 1
                    # print(f"Structure being added{self._data_train['kw'][i+idx]}")
                    rz = self._model(self._data_train['x'][i+idx])
                    # print(rz)
                    if loss is None:
                        # loss = self._loss_func(rz, self._data_train['y'][i+idx])
                        loss = self._loss_func(rz, self._data_train['y'][i+idx], self._data_train['second_dist_matrix'][i+idx])
                    else:
                        # loss += self._loss_func(rz, self._data_train['y'][i+idx])
                        loss += self._loss_func(rz, self._data_train['y'][i+idx], self._data_train['second_dist_matrix'][i+idx])
                # print(loss)
                loss.backward()
                self._optimizer.step()
                log_loss['train'].append(loss.item()/e_size)
            
            self._model.eval()
            for i in range(0, record_num_test, self._batch_size):
                loss = None
                e_size = 0
                for idx in range(self._batch_size):
                    if idx + i >= record_num_test:
                        break
                    e_size += 1
                    rz = self._model(self._data_test['x'][i+idx])
                    if loss is None:
                        # loss = self._loss_func(rz, self._data_test['y'][i+idx])
                        loss = self._loss_func(rz, self._data_test['y'][i+idx], self._data_test['second_dist_matrix'][i+idx])
                    else:
                        # loss += self._loss_func(rz, self._data_test['y'][i+idx])
                        loss += self._loss_func(rz, self._data_test['y'][i+idx], self._data_test['second_dist_matrix'][i+idx])
                        
                log_loss['test'].append(loss.item()/e_size)
            # sys.stderr.write(f'\r{time.asctime()} Test Progress: {len(Xv)} - {len(Xv)}\n')
            self.log['epoch'].append(epoch+1)
            self.log['train'].append(np.mean(log_loss["train"]))
            self.log['test'].append(np.mean(log_loss["test"]))
            # self.log['lr'].append(scheduler.get_last_lr()[0])
            # scheduler.step(np.mean(log_loss["test"]))
            
            if self._show_progress:
                sys.stderr.write(f'\r{time.asctime()} epoch: {epoch} Train Loss (mean): {np.mean(log_loss["train"]):.6f} Test Loss (mean): {np.mean(log_loss["test"]):.6f}')
            
            if (epoch+1) % self._save_span == 0:
                torch.save(self._model.state_dict(), os.path.join(f'{self._output_path}', f'{epoch+1}.pth'))
                 
        sys.stderr.write('\nDone\n')
        self._trained = True
    
    def plot_loss_curve(self, figsize=(10,8)):
        fig = plt.figure(figsize=figsize)
        # Determine when to start logging loss, for models using orig encoder prefer a higher value, ex 25, otherwise 5 is OK
        plt.plot(self.log['epoch'][5:], self.log['train'][5:],  'ro-', label='train')
        plt.plot(self.log['epoch'][5:], self.log['test'][5:],  'go-', label='test')
        plt.ylabel('Loss')
        plt.xlabel('Epoch')
        plt.legend()
        plt.savefig(os.path.join(self._output_path, 'loss_curve.png'))
        
# definition of models
class ModelBase(nn.Module):
    def __init__(self, params: ModelParams, layers: list):
        
        """
        tgt_scope must be a uneven number so the model can look same scope of both side of the target site.
        """
        super(ModelBase, self).__init__()
        
        self._params = params
        self._layers = layers
        self.embedding = nn.Embedding(params.src_vocb_size, params.d_model) 
        self.capture_features = nn.ModuleList(self.get_layers())
        
        # normaly, final and activation should be redefined when derivating a class from this base class
        self.final = nn.Linear(params.d_model, params.tgt_vocb_size)
        self.activation = nn.ReLU()

    def get_layers(self):
        global lidx_layers
        return [lidx_layers[i](self._params) for i in self._layers]


class NT3D(ModelBase):
    
    def __init__(self, params: ModelParams, layers: list):
        super(NT3D, self).__init__(params, layers)
        
        self.final = nn.Linear(params.d_model, params.tgt_vocb_size)
        # self.arpe = AsynRotationalPositionalEncoding(params)
        self.activation = nn.ReLU()
        # self.initialize_parameters()
        
    def forward(self, src):
        src_embedded = self.embedding(src)
        enc_output = src_embedded
        for cf_layer in self.capture_features:     
            enc_output = cf_layer(enc_output)

        residules = enc_output.unsqueeze(2) * enc_output.unsqueeze(1)
        residules = self.activation(self.final(residules))
        return residules[0].view(residules.shape[1], residules.shape[2])
    
    def initialize_parameters(self):
        # Initialize weights and biases
        for layer in self.children():
            if isinstance(layer, nn.Linear):
                torch.nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')  # He initialization
                torch.nn.init.zeros_(layer.bias)  # Initialize biases to zero

class NT3Dv1(ModelBase):
    
    def __init__(self, params: ModelParams, layers: list):
        super(NT3Dv1, self).__init__(params, layers)
        
        self.final = nn.Linear(params.d_model, params.tgt_vocb_size)
        # self.arpe = AsynRotationalPositionalEncoding(params)
        self.activation = nn.ReLU()
        # self.initialize_parameters()
        
    def forward(self, src):
        src_embedded = self.embedding(src)
        enc_output = src_embedded
        # print(f"Enc output size: {enc_output.shape}")
        
        for cf_layer in self.capture_features:     
            enc_output = cf_layer(enc_output)
        # print(f"Enc output size after cf_layers: {enc_output.shape}")

        residules = enc_output.unsqueeze(2) * enc_output.unsqueeze(1)
        # print(f"residules outut shape: {residules.shape}")
        residules = self.activation(self.final(residules))
        # print(f"Residules shape after activation: {residules.shape}")
        final_output = residules[0].view(residules.shape[1], residules.shape[2])
        # print(f"Final output shape: {final_output.shape}")
        return residules[0].view(residules.shape[1], residules.shape[2])
    
    def initialize_parameters(self):
        # Initialize weights and biases
        for layer in self.children():
            if isinstance(layer, nn.Linear):
                torch.nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')  # He initialization
                torch.nn.init.zeros_(layer.bias)
print("Model pipeline loaded")

# Check device used
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}, qty: {torch.cuda.device_count()}")

In [2]:
def tokenization(seqs, max_len=210):
    
    token_dict = {
        'DNA': {'A': 1, 'T': 2, 'C': 3, 'G': 4, 'U': 6, 'S':9, 'P':10},
        'RNA': {'A': 5, 'U': 6, 'C': 7, 'G': 8, 'S':9, 'P':10}
    }

    def detect_type(seq):
        if 'T' in seq and 'U' not in seq:
            return 'DNA'
        elif 'T' not in seq and 'U' in seq:
            return 'RNA'
        else:
            return 'DNA'

    tokens = []
    for seq in seqs:
        seq = seq.upper()
        nt_type = detect_type(seq)
        residules = ''.join([f'PS{base}' for base in seq])[1:]
        tokens += [token_dict[nt_type][resi] for resi in residules]

    # tokens += [0 for _ in range(max_len- len(tokens))]
    return tokens

class NucleotideStructure:

    maxi_length = 320
    # maxi_length = 95

    @property
    def tokens(self):
        return tokenization(self.seqs, len(self.residules))

    @property
    def distance_matrix(self):
        coords = self.coordinates
        size = len(self.residules)
        d_matrix = np.zeros((size, size)) -1
        for i in range(coords.shape[0]):
            for j in range(i, coords.shape[0]):
                d_matrix[i,j] = np.sqrt(np.sum((coords[i,:] - coords[j,:])**2))
                d_matrix[j,i] = d_matrix[i,j]
        return d_matrix
    
    @property
    def scope_mat(self, scope_size=10):
        d_matrix = self.distance_matrix
        L = len(self.residules)
        matrix = np.zeros((L, scope_size*2+1))
        
        for i in range(d_matrix.shape[0]):
            left = np.max([i-scope_size, 0])
            right =  np.min([i+scope_size+1, L])
            left2 = np.max([scope_size - i, 0])
            right2 = np.min([scope_size+L-i, scope_size*2+1])
            matrix[i, left2:right2] = d_matrix[i, left:right]
        return matrix
        
    @property
    def coordinates(self):
        return np.array([self.x, self.y, self.z]).T
    
    # Return sequence to be put in data
    @property
    def sequence(self):
        return self.seqs
    
    # Return orig coordinates to be put in data
    @property
    def x_coord(self):
        return self.x
    @property
    def y_coord(self):
        return self.y
    @property
    def z_coord(self):
        return self.z
    @property
    def residule(self):
        return self.residules 

    def __init__(self, name, seqs, residules, x, y, z, p=0.2):

        self.name = name
        self.seqs = seqs
        self.residules = residules
        self.x = x
        self.y = y
        self.z = z
        self.features = {}
    
    def save(self, output_file):

        with open(output_file, 'w') as fhd:
            json.dump(self.__dict__, fhd)
    
    def split(self, p=0.2):
        if np.random.random() < p:
            self.features['Train'] = False
        else:
            self.features['Train'] = True

    @staticmethod
    def set_maxi_length(maxi_len=200):
        NucleotideStructure.maxi_length = maxi_len
        
def from_json(json_file):
    with open(json_file) as jfile:
        data = json.load(jfile)
        return NucleotideStructure(name=data['name'], seqs=data['seqs'], residules=data['residules'], x=data['x'], y=data['y'], z=data['z'])

def load_data(input_dir, maxi_length, split=False):
    NucleotideStructure.set_maxi_length(maxi_length)

    items = {}
    ifiles = os.listdir(input_dir)
    for ifile in ifiles:
        if os.path.splitext(ifile)[-1] == '.json':
            item = from_json(os.path.join(input_dir, ifile))
            if split:
                item.split()
            # Remove RNA from dataset (if needed)
            # if 'U' in ''.join(item.seqs) and 'T' not in ''.join(item.seqs):
            #     continue
            items[item.name] = item
    # print(items.)
    return items

In [ ]:
def split_string_at_indexes(s, string):
    segments = []
    start = 0
    for segment in s:
        segment_length = len(segment)
        segments.append(string[start:start + segment_length])
        start += segment_length
    return segments

def tokenizing_secondary(secondary_structure):
    tokenizer_ss = {
        ".": 0,
        "+": 1
    }
    tokenized_seq = []
    for strand in secondary_structure:
        # print(strand)
        for token in strand:
            tokenized_seq.append(tokenizer_ss[token])
        # tokenized_seq.append(0) # omitted as there is no reasonable way to force the model to learn strand break
    return torch.tensor(tokenized_seq)

def extract_secondary_structure(pdb_id, sequence, max_len):
    global loop_properties
    row_info = loop_properties[loop_properties['ID'] == pdb_id]
    # print(row_info)
    if row_info.empty:
        # print(f"No G4 structure in {pdb_id}")
        secondary_structure = "." * len("".join(sequence))
        secondary_segments = split_string_at_indexes(sequence, secondary_structure)
    else:
        if row_info["vRNA generation (actual)"].iloc[0] == "N":
            # print(f"No G4 structure in {pdb_id}")
            secondary_structure = "." * len("".join(sequence))
            secondary_segments = split_string_at_indexes(sequence, secondary_structure)
        else:
            secondary_structure = row_info["vRNA generation (actual)"].iloc[0].replace(" ", "")
            secondary_segments = split_string_at_indexes(sequence, secondary_structure)
    # print(secondary_segments)
    tokenized_secondary = tokenizing_secondary(secondary_segments)
    return tokenized_secondary

def get_dot_bracket(pdb_id, sequence):
    row_info = loop_properties[loop_properties['ID'] == pdb_id]
    if row_info.empty:
        return "."*len(sequence)
    elif row_info["vRNA generation (actual)"].iloc[0] == "N":
        return "."*len(sequence)
    else:
        return row_info["vRNA generation (actual)"].iloc[0].replace(" ", "")

def block_cond(loop_order):
    loop_order.remove('Bulge') if 'Bulge' in loop_order else None
    if len(loop_order) // 3 >= 2:
        # ex. 6 loops // 3 -> 2, 9 loops -> 3, etc.
        block_condition = loop_order // 3
        return block_condition
    else:
        block_condition = 1
        return block_condition

def extract_loop_info(pdb_id):
    # print(f"PDB: {pdb_id}")
    row_info = loop_properties[loop_properties['ID'] == pdb_id]
    # print("After extraction")
    if row_info.empty:
        print(f"No G4 structure in {pdb_id}")
        return ['N'], 1
    else:
        loop_info = row_info['Loop order'].iloc[0]
        loop_info = loop_info.split(', ')
        loop_info = [i.replace(' ', '') for i in loop_info]
        block_num = block_cond(loop_order)
        return loop_info, block_num

def asymmetric_cond_check(pdb_id):
    row_info = loop_properties[loop_properties['ID'] == pdb_id]
    if row_info.empty:
        return False
    elif row_info['Asymmetry'].iloc[0] == 'N':
        return False
    else:
        return row_info['Asymmetry'].iloc[0].split(', ')
    
properties = pd.read_csv('./g4_information.csv')
columns = ['ID', 'cg structure success', 'Loop order', 'Strand conformation?', 'Asymmetry', 'vRNA generation (actual)']
loop_properties = properties[columns].copy()
loop_properties['vRNA generation (actual)'] = loop_properties['vRNA generation (actual)'].replace(
    r"[()=]", ".", regex=True
)
print("Loop properties loaded")
# print(loop_properties)
from G4_encoder import get_G4_secondary_structure
from vRNA_encoder import get_secondary_structure

def plot_distmatrix_heatmap(dist_mat, label_str):
    # Create the heatmap using matplotlib
    plt.figure(figsize=(8, 6.5))
    plt.imshow(dist_mat, cmap='viridis', aspect='auto')
    
    # Set the labels for the x and y ticks
    # plt.xticks(ticks=np.arange(len(label_str)), labels=list(label_str))
    # plt.yticks(ticks=np.arange(len(label_str)), labels=list(label_str))
    
    # Add color bar
    plt.colorbar()
    # Add annotations
    for i in range(len(label_str)):
        for j in range(len(label_str)):
            plt.text(j, i, f'{dist_mat[i, j]:.2f}', ha='center', va='center', color='white')
    
    # Set the labels for the axes
    plt.xlabel("Residue")
    plt.ylabel("Residue")
    plt.title("Heatmap with Character Labels")
    
    # Show the plot
    plt.show()


# Example implementation - G4 structure (parallel)
label_string = ['TGAGGGTGGGTAGGGTGGGTAA'] # 1xav
label_secondary = '...+++.+++..+++.+++...'
loop_order = ['Propeller', 'Propeller', 'Propeller']

structure = get_G4_secondary_structure(
    primary_sequence=label_string,
    secondary_structure=label_secondary,
    loop_order=loop_order,
    multi_block_cond=1,
    asymmetric_block=False
)
distance_matrix = structure.gen_distance_matrix()
plot_distmatrix_heatmap(distance_matrix, label_string)
# Example implementation - G4 structure (hybrid)
label_string = ['TAGGGTTAGGGTTAGGGTTAGGGTT'] # 2jsq
label_secondary = '..+++...+++...+++...+++.. '
loop_order = ['Lateral', 'Lateral', 'Propeller']

structure = get_G4_secondary_structure(
    primary_sequence=label_string,
    secondary_structure=label_secondary,
    loop_order=loop_order,
    multi_block_cond=1,
    asymmetric_block=False
)
distance_matrix = structure.gen_distance_matrix()
plot_distmatrix_heatmap(distance_matrix, label_string)
# Example implementation - G4 structure (antiparallel)
label_string = ['GGGTAGGGAGCGGGAGAGGG'] # 6gzn
label_secondary = '++=..++=...=++...++='
loop_order = ['Lateral', 'Diagonal', 'Lateral']

structure = get_G4_secondary_structure(
    primary_sequence=label_string,
    secondary_structure=label_secondary,
    loop_order=loop_order,
    multi_block_cond=1,
    asymmetric_block=False
)
distance_matrix = structure.gen_distance_matrix()
plot_distmatrix_heatmap(distance_matrix, label_string)


# Example implementation - DNA hairpin
test_string = (['CAATCGGATCGAATTCGATCCGATTG']) # model hairpin
structure_test = get_secondary_structure(primary_sequence=test_string)
distance_matrix = structure_test.gen_distance_matrix()
plot_distmatrix_heatmap(distance_matrix, test_string)

# Example implementation - quadruplex-duplex hybrid (QDH)
def combine_matrices(A, B):
    # Find all unique values in A and B that are not 1
    special_values = np.unique(np.concatenate([A.ravel(), B.ravel()]))
    special_values = special_values[special_values != 1]
    result = np.ones_like(A)
    for v in special_values:
        mask = (A == v) | (B == v)
        result[mask] = v
    return result

label_string = ['TTGGGTGGGCGCGAAGCATTCGCGGGGTGGGT'] # 2m93
label_secondary = '..+++.+++...............+++.+++.' 
loop_order = ['Propeller', 'Propeller', 'Propeller']

structure_g4 = get_G4_secondary_structure(
    primary_sequence=label_string,
    secondary_structure=label_secondary,
    loop_order=loop_order,
    multi_block_cond=1,
    asymmetric_block=False
)
structure_duplex = get_secondary_structure(primary_sequence=(label_string))
distance_matrix_g4 = structure_g4.gen_distance_matrix()
distance_matrix_duplex = structure_duplex.gen_distance_matrix()
distance_matrix = combine_matrices(distance_matrix_g4, distance_matrix_duplex)
plot_distmatrix_heatmap(distance_matrix, label_string)

In [ ]:
# Load dataset
OneSeqRecords = load_data('./data', 210)
data = {
    'train': {'x': [], 'y': [], 'kw': [], 'seq': [], 'second_dist_matrix': []},
    'test': {'x': [], 'y': [], 'kw': [], 'seq': [], 'second_dist_matrix': []}
}
structure_count = 0
np.random.seed(seed=228) # usual seed for all models
for kw in OneSeqRecords:
    structure_count += 1
    if np.random.random() < 0.9:
        data['train']['x'].append(torch.tensor(OneSeqRecords[kw].tokens).unsqueeze(0))
        data['train']['y'].append(torch.tensor(OneSeqRecords[kw].distance_matrix, dtype=torch.float))
        data['train']['kw'].append(kw)
        data['train']['seq'].append(OneSeqRecords[kw].sequence)
        vrna_gen = get_secondary_structure(OneSeqRecords[kw].sequence)
        loop_information, block_number = extract_loop_info(kw)
        g4_gen = get_G4_secondary_structure(OneSeqRecords[kw].sequence,
                                            get_dot_bracket(kw,OneSeqRecords[kw].sequence),
                                            loop_information,
                                            block_number,
                                            asymmetric_cond_check(kw))
        data['train']['second_dist_matrix'].append(torch.tensor(combine_matrices(vrna_gen.gen_distance_matrix(),g4_gen.gen_distance_matrix())))
        print(f"Structure {kw} added to train, structure_count is {structure_count}")
    else:
        data['test']['x'].append(torch.tensor(OneSeqRecords[kw].tokens).unsqueeze(0))
        data['test']['y'].append(torch.tensor(OneSeqRecords[kw].distance_matrix, dtype=torch.float))
        data['test']['kw'].append(kw)
        data['test']['seq'].append(OneSeqRecords[kw].sequence)
        vrna_gen = get_secondary_structure(OneSeqRecords[kw].sequence)
        loop_information, block_number = extract_loop_info(kw)
        g4_gen = get_G4_secondary_structure(OneSeqRecords[kw].sequence,
                                            get_dot_bracket(kw,OneSeqRecords[kw].sequence),
                                            loop_information,
                                            block_number,
                                            asymmetric_cond_check(kw))
        data['test']['second_dist_matrix'].append(torch.tensor(combine_matrices(vrna_gen.gen_distance_matrix(),g4_gen.gen_distance_matrix())))
        print(f"Structure {kw} added to test, structure_count is {structure_count}")
print("Data loaded")
print(len(data['train']['kw']), len(data['test']['kw']))

In [ ]:
def MSE_LOSS_upper(x, tgt):
    upper_diagonal_tensor = torch.ones(x.shape[0], x.shape[0])
    upper_diagonal_tensor = torch.triu(upper_diagonal_tensor, 1)
    rz = torch.sum(upper_diagonal_tensor * (x - tgt) ** 2) / torch.sum(upper_diagonal_tensor)
    return rz

def MSE_LOSS_no_diagnol(x, tgt):
    upper_diagonal_tensor = torch.ones(x.shape[0], x.shape[0])
    torch.fill_diagonal_(upper_diagonal_tensor, 0)
    rz = torch.sum(upper_diagonal_tensor * (x - tgt) ** 2) / torch.sum(upper_diagonal_tensor)
    return rz

# torch.fill_diagonal_ doesn't work in this version for some reason
def MSE_LOSS_no_diagonal(x, tgt):
    # Create an upper diagonal tensor with ones
    upper_diagonal_tensor = torch.ones(x.shape[0], x.shape[0])
    
    # Fill the diagonal with zeros manually
    for i in range(upper_diagonal_tensor.shape[0]):
        upper_diagonal_tensor[i, i] = 0

    # Calculate the mean squared error excluding the diagonal
    rz = torch.sum(upper_diagonal_tensor * (x - tgt) ** 2) / torch.sum(upper_diagonal_tensor)
    return rz

def weighted_MSE_LOSS_no_diagonal(x, tgt, weight):
    # Create an upper diagonal tensor with ones
    upper_diagonal_tensor = torch.ones(x.shape[0], x.shape[0])
    
    # Fill the diagonal with zeros manually
    for i in range(upper_diagonal_tensor.shape[0]):
        upper_diagonal_tensor[i, i] = 0

    # Calculate the mean squared error excluding the diagonal
    rz = torch.sum(upper_diagonal_tensor * (x - tgt) ** 2 * weight) / torch.sum(upper_diagonal_tensor)
    return rz

params = get_params('./config_128.ini')
model2 = NT3Dv1(params, layers=[2, 2, 2, 3, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 1])
optimizer2 = torch.optim.Adadelta(model2.parameters(), lr=0.04)
data_version = 'V9'
print("Model set")

In [ ]:
# Load a pre-existing model for prediction or further training
params = get_params('./config_128.ini')
model2 = NT3Dv1(params, layers=[2, 2, 2, 3, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 1]) 
optimizer2 = torch.optim.Adadelta(model2.parameters(), lr=0.04)
data_version = 'V9'
model2.load_state_dict(torch.load('./pthes/' + data_version + '/3000.pth', weights_only=True))
print(f"Model set (for prediction), version: {data_version}")

In [ ]:
# Train model
print(f"Data version: {data_version}")
exp1 = TrainModel(model=model2, 
                 loss_func=weighted_MSE_LOSS_no_diagonal, 
                 optimizer=optimizer2, 
                 epoch_num=3000,
                 batch_size=64,
                 params_path='./pthes/' + data_version, 
                 save_span=50, 
                 data_train=data['train'], 
                 data_test=data['test'], 
                 show_progress=True
                )
exp1.train()
exp1.plot_loss_curve()

In [ ]:
print(f"Data version: {data_version}")
def MDS(dismat, n=2):
    D = dismat ** 2
    B = double_centering(D)
    Dm, Em = top_features(B, n)
    rz = np.real(np.matmul(Em, Dm ** 0.5))
    return rz

def double_centering(mat):
    size = mat.shape[0]
    C = np.identity(size) - np.ones((size, size)) / size
    return -1/2 * np.matmul(np.matmul(C, mat), C)

def top_features(mat, n=2):
    eigen_vector1, eigen_values, eigen_vector2 = np.linalg.svd(mat)
    Dm = np.zeros((n, n))
    np.fill_diagonal(Dm, eigen_values[:n])
    Em = eigen_vector2[:n].T
    return Dm, Em

def rigid_transform_3D(A, B):
    if A.shape == B.shape:
        pass
    else: 
        # Error handle 1: Wrong size
        if A.shape[0] > B.shape[0]:
            large_dim = A.shape[0]
            small_dim = B.shape[0]
        else:
            large_dim = B.shape[0]
            small_dim = A.shape[0]
        
        if A.shape[0] > B.shape[0]:
            A = A[:(large_dim - small_dim)]
        else:
            B = B[:(large_dim - small_dim)]
        print(A.shape)
        print(B.shape)
        if A.shape == B.shape:
            pass
        else:
            print("Conversion failed, second step")
            if A.shape[0] > B.shape[0]:
                large_dim = A.shape[0]
                small_dim = B.shape[0]
            else:
                large_dim = B.shape[0]
                small_dim = A.shape[0]

            if A.shape[0] > B.shape[0]:
                A = A[:(A.shape[0] - (large_dim - small_dim))]
            else:
                B = B[:(B.shape[0] - (large_dim - small_dim))]

    num_rows, num_cols = A.shape

    # find mean column wise
    centroid_A = np.mean(A, axis=0)
    centroid_B = np.mean(B, axis=0)

    # ensure centroids are 3x1
    centroid_A = centroid_A.reshape(-1, 1)
    centroid_B = centroid_B.reshape(-1, 1)

    # subtract mean
    Am = A - centroid_A.T
    Bm = B - centroid_B.T
    H = Am.T @ Bm
    # find rotation
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    t = -R @ centroid_A + centroid_B

    return R, t

def get_RMSD(c1, c2):
    a, b= rigid_transform_3D(c1, c2)
    # print(a.shape, b.shape)
    try:
        return np.sqrt(np.sum((c1 @ a.T + b.T - c2) ** 2) / c1.shape[0])
    except ValueError:
        # Error handle 1: Wrong size
        if c1.shape[0] > c2.shape[0]:
            large_dim = c1.shape[0]
            small_dim = c2.shape[0]
        else:
            large_dim = c2.shape[0]
            small_dim = c1.shape[0]
        
        if c1.shape[0] > c2.shape[0]:
            c1 = c1[:(large_dim - small_dim)]
        else:
            c2 = c2[:(large_dim - small_dim)]
        print(c1.shape)
        print(c2.shape)
        # assert c1.shape == c2.shape, 'Pred and orig structure arent the same size (RMSD)'
        if c1.shape == c2.shape:
            pass
        else:
            print("Conversion failed, second step")
            if c1.shape[0] > c2.shape[0]:
                large_dim = c1.shape[0]
                small_dim = c2.shape[0]
            else:
                large_dim = c2.shape[0]
                small_dim = c1.shape[0]

            if c1.shape[0] > c2.shape[0]:
                c1 = c1[:(c1.shape[0] - (large_dim - small_dim))]
            else:
                c2 = c2[:(c2.shape[0] - (large_dim - small_dim))]
        return np.sqrt(np.sum((c1 @ a.T + b.T - c2) ** 2) / c1.shape[0])

def write_pdb(filename, atom_names, residue_names, resi_idx, connections, coordinates, chain_id='A'):
    with open(filename, 'w') as f:
        # print(len(atom_names))
        # print(len(residue_names))
        for i in range(len(atom_names)):
            atom_name, residue_name, res_seq, coord = atom_names[i], residue_names[i], resi_idx[i], coordinates[i]
            serial = i + 1
            line = f'ATOM  {serial:5d} {atom_name:<4} {residue_name:<3} {chain_id:<1}{res_seq:4d}    {coord[0]:8.3f}{coord[1]:8.3f}{coord[2]:8.3f}{1:6.2f}{0:6.2f}          {atom_name:>2}  \n'
            f.write(line)
        for bond in connections:
            f.write(f"CONECT{bond[0]:5d}{bond[1]:5d}\n")
        f.write("END\n")

seq_example = 'TTAGGTGGGTAGGGTGGGTGTG' # Q3-sbl2; source: https://academic.oup.com/nar/article/50/20/11948/6842908

tk = torch.tensor(tokenization([seq_example])).unsqueeze(0)

dm = model2(tk).detach().numpy()
np.fill_diagonal(dm, 0)
c1 = MDS(dm, 3)

resi = ''.join([f'PS{b}' for b in seq_example])
bases = []
res_idx = []
connections = []
for i in range(0, len(resi)//3):
    bases += [f'{resi[3*i+2]}', f'{resi[3*i+2]}', f'{resi[3*i+2]}']
    res_idx += [i+1, i+1, i+1]
    cidx = 3 * i
    connections += [(cidx, cidx+1), (cidx+1, cidx+2), (cidx+1, cidx+3)]
write_pdb(f'example.pdb', resi[1:], bases[1:], res_idx[1:], connections[1:-1], c1)
plt.imshow(dm)
# When visualizing PDB file on Pymol, type the command "show sticks" in the Terminal to show the full CG structure